In [4]:
# ============================================================
# TÜRK MARKET SATIŞ VERİSİ — ANOMALİ TESPİTİ + DEĞERLENDİRME
# Ensemble Model + Otomatik Performans Analizi
# ============================================================

import os
import glob
import warnings
import pandas as pd
import numpy as np
import kagglehub
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import DBSCAN
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

print("=" * 70)
print("Türk Market Satış — Anomali Tespiti + Model Değerlendirmesi")
print("=" * 70)

# ============================================================
# BÖLÜM 1: VERİ YÜKLEME VE ÖN İŞLEME
# ============================================================

print("\n[1/9] Veri indiriliyor...")
dataset_path = kagglehub.dataset_download("omercolakoglu/turkish-market-sales-dataset-with-9000items")
excel_files = glob.glob(os.path.join(dataset_path, "**/*.xls*"), recursive=True)

if not excel_files:
    raise FileNotFoundError("Excel dosyası bulunamadı!")

print(f"Okunuyor: {os.path.basename(excel_files[0])}")
df = pd.read_excel(excel_files[0])
print(f"Toplam ham satır: {len(df):,}")

# Sütun eşleştirme
df.rename(columns={
    'DATE_': 'tarih',
    'CLIENTCODE': 'musteri_id',
    'LINENETTOTAL': 'toplam_tutar',
    'AMOUNT': 'miktar',
    'PRICE': 'birim_fiyat',
    'CATEGORY_NAME1': 'kategori',
    'ITEMNAME': 'urun_adi'
}, inplace=True)

df.dropna(subset=['musteri_id'], inplace=True)

# ============================================================
# BÖLÜM 2: ÖZELLİK MÜHENDİSLİĞİ
# ============================================================

print("\n[2/9] Özellik mühendisliği...")
df['tarih'] = pd.to_datetime(df['tarih'])
df['saat'] = df['tarih'].dt.hour
df['gun'] = df['tarih'].dt.day
df['ay'] = df['tarih'].dt.month
df['gece_saati'] = df['saat'].between(2, 5).astype(int)
df['hafta_sonu'] = df['tarih'].dt.dayofweek.isin([5, 6]).astype(int)

musteri_features = df.groupby('musteri_id').agg({
    'toplam_tutar': ['sum', 'mean', 'std', 'count'],
    'miktar': ['sum', 'mean'],
    'birim_fiyat': ['mean', 'max', 'min'],
    'kategori': lambda x: x.nunique(),
    'urun_adi': lambda x: x.nunique(),
    'tarih': lambda x: (x.max() - x.min()).days,
    'gece_saati': 'sum',
    'hafta_sonu': 'mean'
}).reset_index()

musteri_features.columns = ['musteri_id', 'toplam_harcama', 'ort_harcama', 'harcama_std',
                             'alisveris_sayisi', 'toplam_urun', 'ort_urun_miktari',
                             'ort_birim_fiyat', 'max_birim_fiyat', 'min_birim_fiyat',
                             'kategori_cesitliligi', 'urun_cesitliligi', 'aktif_gun_sayisi',
                             'gece_alisverisi', 'hafta_sonu_orani']

musteri_features['gunluk_alisveris'] = musteri_features['alisveris_sayisi'] / (musteri_features['aktif_gun_sayisi'] + 1)
musteri_features['sepet_buyuklugu'] = musteri_features['toplam_urun'] / musteri_features['alisveris_sayisi']
print(f'✅ {musteri_features.shape[0]:,} müşteri için özellikler hazır')

# ============================================================
# BÖLÜM 3: MODEL EĞİTİMİ
# ============================================================

print("\n[3/9] Ensemble modelleri eğitiliyor...")
feature_cols = ['toplam_harcama', 'ort_harcama', 'harcama_std', 'alisveris_sayisi',
                'sepet_buyuklugu', 'gunluk_alisveris', 'gece_alisverisi',
                'ort_birim_fiyat', 'max_birim_fiyat', 'kategori_cesitliligi',
                'urun_cesitliligi']

X = musteri_features[feature_cols].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Model 1: Isolation Forest
iso_forest = IsolationForest(contamination=0.05, random_state=42, n_estimators=100, n_jobs=-1)
iso_predictions = iso_forest.fit_predict(X_scaled)
musteri_features['iso_anomali'] = iso_predictions
iso_count = (iso_predictions == -1).sum()
print(f'  🤖 Isolation Forest: {iso_count:,} anomali')

# Model 2: LOF
lof = LocalOutlierFactor(contamination=0.05, n_neighbors=20, n_jobs=-1)
lof_predictions = lof.fit_predict(X_scaled)
musteri_features['lof_anomali'] = lof_predictions
lof_count = (lof_predictions == -1).sum()
print(f'  🤖 LOF: {lof_count:,} anomali')

# Model 3: DBSCAN
dbscan = DBSCAN(eps=1.5, min_samples=5, n_jobs=-1)
dbscan_labels = dbscan.fit_predict(X_scaled)
musteri_features['dbscan_anomali'] = (dbscan_labels == -1).astype(int) * -1 + (dbscan_labels != -1).astype(int)
dbscan_count = (dbscan_labels == -1).sum()
print(f'  🤖 DBSCAN: {dbscan_count:,} anomali')

# ============================================================
# BÖLÜM 4: ENSEMBLE OYLAMASI
# ============================================================

print('\n[4/9] Ensemble oylaması...')
musteri_features['anomali_vote'] = (
    (musteri_features['iso_anomali'] == -1).astype(int) +
    (musteri_features['lof_anomali'] == -1).astype(int) +
    (musteri_features['dbscan_anomali'] == -1).astype(int)
)

musteri_features['final_anomali'] = (musteri_features['anomali_vote'] >= 2).astype(int)
final_count = musteri_features['final_anomali'].sum()
print(f'🎯 Final karar: {final_count:,} anomali müşteri ({final_count/len(musteri_features)*100:.2f}%)')

# Anomali tipi belirleme
def anomali_tipi_belirle(row):
    tipler = []
    if row['sepet_buyuklugu'] > musteri_features['sepet_buyuklugu'].quantile(0.95):
        tipler.append('Büyük Sepet')
    if row['ort_birim_fiyat'] > musteri_features['ort_birim_fiyat'].quantile(0.95):
        tipler.append('Lüks Alışveriş')
    if row['gece_alisverisi'] > 0:
        tipler.append('Gece Alışverişi')
    if row['gunluk_alisveris'] > musteri_features['gunluk_alisveris'].quantile(0.95):
        tipler.append('Sık Alışveriş')
    return ', '.join(tipler) if tipler else 'Kompleks Anomali'

anomali_musteriler = musteri_features[musteri_features['final_anomali'] == 1].copy()
anomali_musteriler['anomali_tipi'] = anomali_musteriler.apply(anomali_tipi_belirle, axis=1)

# ============================================================
# BÖLÜM 5: MODEL DEĞERLENDİRME - UYUM ANALİZİ
# ============================================================

print("\n" + "=" * 70)
print("MODEL PERFORMANS DEĞERLENDİRMESİ")
print("=" * 70)

print("\n[5/9] Model Uyum Analizi")
print("-" * 70)

# Oy dağılımı
print("\n📊 Ensemble Oy Dağılımı:")
vote_dist = musteri_features['anomali_vote'].value_counts().sort_index()
for oy, sayi in vote_dist.items():
    print(f"  {oy} model anomali dedi: {sayi:,} müşteri ({sayi/len(musteri_features)*100:.2f}%)")

# Model kesişimleri
iso_ve_lof = ((musteri_features['iso_anomali'] == -1) & (musteri_features['lof_anomali'] == -1)).sum()
iso_ve_dbscan = ((musteri_features['iso_anomali'] == -1) & (musteri_features['dbscan_anomali'] == -1)).sum()
lof_ve_dbscan = ((musteri_features['lof_anomali'] == -1) & (musteri_features['dbscan_anomali'] == -1)).sum()
uclu_kesisim = ((musteri_features['iso_anomali'] == -1) & 
                (musteri_features['lof_anomali'] == -1) & 
                (musteri_features['dbscan_anomali'] == -1)).sum()

print("\n🔗 Model Kesişimleri:")
print(f"  IF ∩ LOF       : {iso_ve_lof:,}")
print(f"  IF ∩ DBSCAN    : {iso_ve_dbscan:,}")
print(f"  LOF ∩ DBSCAN   : {lof_ve_dbscan:,}")
print(f"  Üçü birden (⭐): {uclu_kesisim:,} (en güvenilir anomaliler)")

# ============================================================
# BÖLÜM 6: KARAKTERİSTİK ANALİZİ
# ============================================================

print("\n[6/9] Anomali vs Normal Karşılaştırması")
print("-" * 70)

musteri_features['grup'] = musteri_features['final_anomali'].map({0: 'Normal', 1: 'Anomali'})
key_features = ['toplam_harcama', 'ort_harcama', 'alisveris_sayisi', 
                'sepet_buyuklugu', 'gunluk_alisveris', 'kategori_cesitliligi']

print(f"\n{'Özellik':<25} {'Normal':<15} {'Anomali':<15} {'Fark (%)':<10}")
print("-" * 70)

for feature in key_features:
    normal_mean = musteri_features[musteri_features['grup'] == 'Normal'][feature].mean()
    anomali_mean = musteri_features[musteri_features['grup'] == 'Anomali'][feature].mean()
    fark_pct = ((anomali_mean - normal_mean) / normal_mean * 100) if normal_mean != 0 else 0
    print(f"{feature:<25} {normal_mean:>14,.2f} {anomali_mean:>14,.2f} {fark_pct:>9.1f}%")

# ============================================================
# BÖLÜM 7: İSTATİSTİKSEL ANLAMLILIK
# ============================================================

print("\n[7/9] İstatistiksel Anlamlılık Testleri")
print("-" * 70)
print("(Mann-Whitney U Test - p < 0.001 çok anlamlı)\n")

for feature in key_features:
    normal_data = musteri_features[musteri_features['grup'] == 'Normal'][feature].dropna()
    anomali_data = musteri_features[musteri_features['grup'] == 'Anomali'][feature].dropna()
    
    statistic, p_value = stats.mannwhitneyu(normal_data, anomali_data, alternative='two-sided')
    anlamli = "✅ ÇOK ANLAMLI" if p_value < 0.001 else "⚠️  Anlamlı" if p_value < 0.05 else "❌ Anlamsız"
    print(f"{feature:<25} p = {p_value:.2e}  {anlamli}")

# ============================================================
# BÖLÜM 8: ŞİDDET SKORU HESAPLAMA
# ============================================================

print("\n[8/9] Anomali Şiddet Skorları")
print("-" * 70)

severity_features = ['toplam_harcama', 'sepet_buyuklugu', 'gunluk_alisveris', 
                     'ort_birim_fiyat', 'gece_alisverisi']

z_scores = []
for feature in severity_features:
    mean_val = musteri_features[feature].mean()
    std_val = musteri_features[feature].std()
    anomali_musteriler[f'{feature}_z'] = np.abs((anomali_musteriler[feature] - mean_val) / std_val)
    z_scores.append(f'{feature}_z')

anomali_musteriler['severity_score'] = anomali_musteriler[z_scores].mean(axis=1)

print("\n🔴 En Şiddetli 10 Anomali:\n")
top_10 = anomali_musteriler.nlargest(10, 'severity_score')[
    ['musteri_id', 'severity_score', 'toplam_harcama', 'alisveris_sayisi', 'anomali_tipi']
]
print(top_10.to_string(index=False))

# Şiddet kategorisi
anomali_musteriler['severity_level'] = pd.cut(
    anomali_musteriler['severity_score'], 
    bins=[0, 2, 4, np.inf], 
    labels=['Düşük', 'Orta', 'Yüksek']
)

print("\n📊 Şiddet Dağılımı:")
print(anomali_musteriler['severity_level'].value_counts().sort_index())

# ============================================================
# BÖLÜM 9: PERFORMANS METRİKLERİ
# ============================================================

print("\n[9/9] Genel Model Metrikleri")
print("=" * 70)

# Contamination karşılaştırması
expected_cont = 0.05
actual_cont = final_count / len(musteri_features)
print(f"\n📌 Kontaminasyon Kontrolü:")
print(f"  Hedeflenen  : %{expected_cont*100:.1f}")
print(f"  Gerçekleşen : %{actual_cont*100:.2f}")
print(f"  Sapma       : %{abs(expected_cont - actual_cont)*100:.2f}")

# Konsensüs oranı
consensus_rate = uclu_kesisim / final_count if final_count > 0 else 0
print(f"\n🎯 Model Güvenilirlik Skoru:")
print(f"  Konsensüs Oranı: %{consensus_rate*100:.1f}")
print(f"  Yorum: {'Yüksek güven' if consensus_rate > 0.5 else 'Orta güven' if consensus_rate > 0.3 else 'Düşük güven'}")

# Effect size
print(f"\n💪 Ayırıcılık Gücü (Cohen's d):")
for feature in key_features[:3]:
    normal_data = musteri_features[musteri_features['grup'] == 'Normal'][feature]
    anomali_data = musteri_features[musteri_features['grup'] == 'Anomali'][feature]
    
    pooled_std = np.sqrt((normal_data.var() + anomali_data.var()) / 2)
    cohens_d = (anomali_data.mean() - normal_data.mean()) / pooled_std if pooled_std != 0 else 0
    
    guc = "🟢 Güçlü" if abs(cohens_d) > 0.8 else "🟡 Orta" if abs(cohens_d) > 0.5 else "🔴 Zayıf"
    print(f"  {feature:<25}: d = {cohens_d:>6.2f}  {guc}")

# Anomali tipi dağılımı
print(f"\n📊 Tespit Edilen Anomali Profilleri:")
tip_dagilim = anomali_musteriler['anomali_tipi'].value_counts()
for tip, sayi in tip_dagilim.items():
    print(f"  {tip:<30}: {sayi:>6,} ({sayi/len(anomali_musteriler)*100:>5.1f}%)")

# ============================================================
# SONUÇLARI KAYDET
# ============================================================

print("\n" + "=" * 70)
print("DOSYALAR KAYDEDİLİYOR...")
print("=" * 70)

musteri_features.to_csv('anomali_tum_musteriler.csv', index=False, encoding='utf-8-sig')
anomali_musteriler.to_csv('anomali_musteriler_skorlu.csv', index=False, encoding='utf-8-sig')

# Özet rapor
ozet = {
    'Toplam Müşteri': len(musteri_features),
    'Anomali Sayısı': final_count,
    'Anomali Oranı (%)': round(actual_cont * 100, 2),
    'IF Anomali': iso_count,
    'LOF Anomali': lof_count,
    'DBSCAN Anomali': dbscan_count,
    'Üçlü Konsensüs': uclu_kesisim,
    'Konsensüs Oranı (%)': round(consensus_rate * 100, 1),
    'En Yaygın Tip': tip_dagilim.index[0] if len(tip_dagilim) > 0 else 'N/A'
}

pd.DataFrame([ozet]).to_csv('model_performans_ozeti.csv', index=False, encoding='utf-8-sig')

print("\n✅ Kaydedilen dosyalar:")
print("  1. anomali_tum_musteriler.csv - Tüm müşteriler + tahminler")
print("  2. anomali_musteriler_skorlu.csv - Anomaliler + şiddet skoru")
print("  3. model_performans_ozeti.csv - Performans metrikleri")

print("\n" + "=" * 70)
print("ANALİZ TAMAMLANDI! ✅")
print("=" * 70)

print("\n📝 Sonuç Yorumu:")
print("  • Modelin güvenilirliği için konsensüs oranına bakın (>%50 ideal)")
print("  • Tüm özellikler istatistiksel olarak anlamlıysa model çalışıyor")
print("  • Yüksek şiddet skorlu müşteriler öncelikli incelenmeli")
print("  • DBSCAN'in az anomali bulması normal (yoğunluk tabanlı çalışır)")
print("=" * 70)

Türk Market Satış — Anomali Tespiti + Model Değerlendirmesi

[1/9] Veri indiriliyor...
Okunuyor: MarketSales.xlsx
Toplam ham satır: 611,108

[2/9] Özellik mühendisliği...
✅ 49,316 müşteri için özellikler hazır

[3/9] Ensemble modelleri eğitiliyor...
  🤖 Isolation Forest: 2,466 anomali
  🤖 LOF: 2,466 anomali
  🤖 DBSCAN: 284 anomali

[4/9] Ensemble oylaması...
🎯 Final karar: 488 anomali müşteri (0.99%)

MODEL PERFORMANS DEĞERLENDİRMESİ

[5/9] Model Uyum Analizi
----------------------------------------------------------------------

📊 Ensemble Oy Dağılımı:
  0 model anomali dedi: 44,760 müşteri (90.76%)
  1 model anomali dedi: 4,068 müşteri (8.25%)
  2 model anomali dedi: 316 müşteri (0.64%)
  3 model anomali dedi: 172 müşteri (0.35%)

🔗 Model Kesişimleri:
  IF ∩ LOF       : 376
  IF ∩ DBSCAN    : 283
  LOF ∩ DBSCAN   : 173
  Üçü birden (⭐): 172 (en güvenilir anomaliler)

[6/9] Anomali vs Normal Karşılaştırması
----------------------------------------------------------------------

Özelli